In [ ]:
import pandas as pd
import plotly.express as px
import geopandas as gpd
import plotly.graph_objects as go
import json
import numpy as np 

In [ ]:
df_adresse = pd.read_csv('/home/jovyan/work/canc_air/01_data/data_octobre_2023/Pseudonymisation_provisoire_geocoded.csv', sep = ";")
df_adresse.drop('Unnamed: 0', axis=1, inplace=True)

df_clinique = pd.read_excel("/home/jovyan/work/canc_air/01_data/data_octobre_2023/pseudonymisation_id_sexe_ddn_loc.xlsx")


In [ ]:
##### Données du dataframe departement  
df_dept = gpd.read_file('/home/jovyan/work/canc_air/01_data/departements/DEPARTEMENT.shp')

## Transformation du crs du dataframe WGS84 vers Lambert93 
df_dept = (df_dept.set_crs(epsg=2154)).to_crs(epsg=4326)

### Jointure des deux datasets pour obtenir la provenance des patients dans le df adresse 

In [ ]:
df_adresse_clinique = df_adresse.merge(df_clinique, on='pseudo_provisoire', how='left')


In [ ]:
#df_adresse_clinique[['pseudo_provisoire',  'x', 'y', 'geometry', 'centre']].head().to_csv("/home/jovyan/work/canc_air/01_data/test.csv")

In [ ]:
## Check for nan values : 
missing_values = df_adresse_clinique[['x', 'y', 'centre']].isnull().sum()
missing_values

## remove them : 
cleaned_data = df_adresse_clinique.dropna(subset=['x', 'y', 'centre'])


In [ ]:
fig = px.scatter_mapbox(cleaned_data, lat='y', lon='x', color='centre',
                            color_continuous_scale=px.colors.cyclical.IceFire,
                            size_max=15, zoom=5,
                            mapbox_style="carto-positron",
                            labels={'centre': 'centre'},
                            title='Répartition des patients selon leur centre de soin')

fig.add_trace(df_dept
                
fig.update_layout(margin={"r":0,"t":0,"l":0,"b":0}, 
                      width=1200,  # Width of the figure
                        height=600 # Height of the figure
                     )

# Show the plot
fig.show()

In [ ]:
fig = px.scatter_mapbox(cleaned_data, lat='y', lon='x', color='centre',
                            color_continuous_scale=px.colors.cyclical.IceFire,
                            size_max=15, zoom=5,
                            mapbox_style="carto-positron",
                            labels={'centre': 'centre'},
                            title='Répartition des patients selon leur centre de soin')

# Iterate through the geometries and add them as lines to the figure
for geometry in df_dept.geometry:
    # Ensure that the geometry is valid and not empty
    if geometry and not geometry.is_empty:
        if geometry.geom_type == 'Polygon':
            x, y = geometry.exterior.coords.xy
            fig.add_trace(go.Scattergeo(lon=list(x), lat=list(y), mode='lines', line=dict(width=10)))
        elif geometry.geom_type == 'MultiPolygon':
            for polygon in geometry.geoms:
                x, y = polygon.exterior.coords.xy
                fig.add_trace(go.Scattergeo(lon=list(x), lat=list(y), mode='lines', line=dict(width=10, color='rgba(0, 0,0)')))

                
fig.update_layout(margin={"r":0,"t":0,"l":0,"b":0}, 
                      width=1200,  # Width of the figure
                        height=600 # Height of the figure
                     )

# Show the plot
fig.show()

### Zoom sur IDF 

In [ ]:
##### Données du dataframe departement  
df_dept = gpd.read_file('/home/jovyan/work/canc_air/01_data/departements/DEPARTEMENT.shp')

## Transformation du crs du dataframe WGS84 vers Lambert93 
df_dept = (df_dept.set_crs(epsg=2154)).to_crs(epsg=4326)

gdf = (gpd.GeoDataFrame(cleaned_data, geometry=gpd.points_from_xy(cleaned_data.x, cleaned_data.y)).set_crs(epsg=4326))#.to_crs(epsg=2154)


In [ ]:
def spatialjoin(gdf_data, gdf_spatial_layer):
    """Spatial join of the database and the department layers"""
    
    print("Réalisation de la jointure spatiale avec les départements métropolitains...")
    
    df_join_dept = gdf_data.sjoin(gdf_spatial_layer[['CODE_DEPT','geometry']], how="left", predicate='within')
    df_join_dept.drop('index_right', axis=1, inplace=True)
    
    return df_join_dept 

df_join_dept = spatialjoin(gdf, df_dept)


In [ ]:
## suppression des valeurs nan 
df_join_dept = df_join_dept.dropna(subset=['CODE_DEPT'])

##Transformation de la colonne en str pour filtrer 
df_join_dept.CODE_DEPT = df_join_dept.CODE_DEPT.astype(str)


dept_idf = ["75","77", "78", "91", "92", "93", "94", "95"]
df_IDF = df_join_dept.loc[df_join_dept['CODE_DEPT'].isin(dept_idf)] 

In [ ]:
df_IDF_clean = df_IDF.dropna(subset=['x', 'y', 'centre'])


In [ ]:
fig_map = px.scatter_mapbox(df_IDF_clean, lat='y', lon='x', color='centre',
                            color_continuous_scale=px.colors.cyclical.IceFire,
                            size_max=15, zoom=10,
                            mapbox_style="carto-positron",
                            labels={'centre': 'centre'},
                            title='Répartition des patients selon leur centre de soin')

fig_map.update_layout(margin={"r":0,"t":0,"l":0,"b":0}, 
                      width=1200,  # Width of the figure
                        height=600 # Height of the figure
                     )

# Show the plot
fig_map.show()